# MoveNet Thunder · EPFL 무릎 굽힘 평가
Colab에서 모두 실행합니다. EPFL 사진·라벨 ZIP을 공식 서버에서 내려받습니다. 실패하면 기존 ZIP을 선택할 수 있습니다. 모델 입력은 전체 프레임이며 정답 좌표는 평가에만 사용합니다. 작은·큰 그룹은 촬영 거리 대신 화면상 몸통 길이로 구분합니다.

로컬 PC에서도 실행할 수 있습니다. ZIP 두 개와 Kaggle에서 받은 모델 폴더 `movenet-tensorflow2-singlepose-thunder-v4`를 노트북과 같은 폴더에 두면 내려받기 없이 실행합니다.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import io, json, math, zipfile, statistics

names = ['ski2dpose_labels.json.zip', 'ski2dpose_images_webp.zip']
for name in names:
    if not Path(name).is_file():
        try:
            dest = Path(name + '.part')
            urlretrieve('https://datasets-cvlab.epfl.ch/2019-ski-2d-pose/' + name, dest)
            with zipfile.ZipFile(dest) as archive:
                if archive.testzip() is not None:
                    raise ValueError('Invalid ZIP')
            dest.replace(name)
        except Exception as error:
            print(error, 'Upload:', name)
            from google.colab import files
            files.upload()
    assert zipfile.is_zipfile(name), name

In [2]:
def flexion(points):
    hip, knee, ankle = points
    a = (hip[0]-knee[0], hip[1]-knee[1])
    b = (ankle[0]-knee[0], ankle[1]-knee[1])
    if not math.hypot(*a) or not math.hypot(*b):
        return None
    return 180 - math.degrees(math.atan2(abs(a[0]*b[1]-a[1]*b[0]), a[0]*b[0]+a[1]*b[1]))

def webp_size(data):
    kind = data[12:16]
    if kind == b'VP8 ':
        return tuple(int.from_bytes(data[i:i+2], 'little') & 16383 for i in (26,28))
    if kind == b'VP8X':
        return tuple(1 + int.from_bytes(data[i:i+3], 'little') for i in (24,27))
    if kind == b'VP8L':
        bits = int.from_bytes(data[21:25], 'little')
        return 1+(bits & 16383), 1+((bits >> 14) & 16383)
    raise ValueError('Invalid WebP')

with zipfile.ZipFile(names[0]) as archive:
    labels = json.loads(archive.read('ski2dpose_labels.json'))
rows = []
with zipfile.ZipFile(names[1]) as images:
    for video, splits in labels.items():
        for split, records in splits.items():
            for image, record in records.items():
                key = f'Images_webp/{video}/{split}/{image}.webp'
                with images.open(key) as stream:
                    w,h = webp_size(stream.read(32))
                p = [(x*w,y*h,v) for x,y,v in record['annotation']]
                row = {'video':video, 'image':key, 'width':w, 'height':h}
                sizes = [math.dist(p[a][:2],p[b][:2])/h for a,b in [(2,10),(6,13)] if p[a][2] and p[b][2]]
                row['size'] = statistics.mean(sizes) if sizes else None
                for side,start in [('right',10),('left',13)]:
                    leg = p[start:start+3]
                    row['gt_'+side] = flexion(leg) if all(j[2] for j in leg) else None
                rows.append(row)
cut = statistics.median(r['size'] for r in rows if r['size'] is not None)
for r in rows:
    r['group'] = 'unknown' if r['size'] is None else ('small' if r['size'] <= cut else 'large')
from collections import Counter
print('Torso / image height median:', round(cut,4))
print(Counter(r['group'] for r in rows))
print('Both ground-truth knees:',sum(r['gt_left'] is not None and r['gt_right'] is not None for r in rows))

Torso / image height median: 0.0925
Counter({'large': 960, 'small': 960, 'unknown': 62})
Both ground-truth knees: 1527


In [3]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image

# Kaggle에서 받은 SavedModel 폴더가 노트북 옆에 있으면 그것을 쓰고, 없으면 TF Hub에서 내려받습니다.
model_dir = Path(os.environ.get('MOVENET_DIR', 'movenet-tensorflow2-singlepose-thunder-v4'))
if (model_dir/'saved_model.pb').is_file():
    model = tf.saved_model.load(str(model_dir))
    print('Model:', model_dir.name)
else:
    import tensorflow_hub as hub
    model = hub.load('https://tfhub.dev/google/movenet/singlepose/thunder/4')
    print('Model: tfhub.dev/google/movenet/singlepose/thunder/4')
infer = model.signatures['serving_default']
print('TensorFlow', tf.__version__, '| input', infer.structured_input_signature[1])
with zipfile.ZipFile(names[1]) as images:
    for idx,r in enumerate(rows):
        rgb = np.array(Image.open(io.BytesIO(images.read(r['image']))).convert('RGB'))
        h,w = rgb.shape[:2]
        scale = 256/max(w,h)
        nw,nh = max(1,round(w*scale)),max(1,round(h*scale))
        x0,y0 = (256-nw)//2,(256-nh)//2
        resized = tf.image.resize(rgb,[nh,nw])
        padded = tf.image.pad_to_bounding_box(resized,y0,x0,256,256)
        out = infer(tf.cast(padded[None],tf.int32))['output_0'].numpy()[0,0]
        for side,ids in [('left',[11,13,15]),('right',[12,14,16])]:
            leg = out[ids]
            points = [((x*256-x0)*w/nw,(y*256-y0)*h/nh) for y,x,score in leg]
            r['pred_'+side] = flexion(points) if np.all(leg[:,2]>=0.30) else None
        if (idx+1)%100==0:
            print(idx+1,'/',len(rows))

Model: movenet-tensorflow2-singlepose-thunder-v4
TensorFlow 2.21.0 | input {'input': TensorSpec(shape=(1, 256, 256, 3), dtype=tf.int32, name='input')}


100 / 1982


200 / 1982


300 / 1982


400 / 1982


500 / 1982


... (10 outputs left out)

1600 / 1982


1700 / 1982


1800 / 1982


1900 / 1982


In [4]:
df = pd.DataFrame(rows)
df.to_csv('movenet_frames.csv',index=False)
summary = []
for name,g in [('all',df),*df.groupby('group')]:
    eligible = g[['gt_left','gt_right']].notna().all(axis=1)
    both = g[['pred_left','pred_right']].notna().all(axis=1)
    result = dict(group=name,frames=len(g),both_pred=int(both.sum()),coverage_pct=100*both.mean(),
                  both_gt=int(eligible.sum()),both_paired=int((eligible & both).sum()))
    for side in ['left','right']:
        errors = (g['gt_'+side]-g['pred_'+side]).abs().dropna()
        result[side+'_mae_deg'] = errors.mean()
        result[side+'_pairs'] = len(errors)
    summary.append(result)
report = pd.DataFrame(summary)
report.to_csv('movenet_summary.csv',index=False)
display(report)

,group,frames,both_pred,coverage_pct,both_gt,both_paired,left_mae_deg,left_pairs,right_mae_deg,right_pairs
0,all,1982,569,28.708375,1527,482,25.354561,741,25.563139,740
1,large,960,297,30.937500,788,268,20.144330,422,21.324237,409
2,small,960,257,26.770833,737,213,32.370431,317,30.877381,329
3,unknown,62,15,24.193548,2,1,12.697677,2,18.225822,2
